# Haystack 3.x Quickstart
Workshop prep notebook. Covers Pipeline, Components, RAG, Agents.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

## 1. Document Store + indexing

In [ ]:
store = InMemoryDocumentStore()
docs = [
    Document(content='Haystack is open-source for LLM apps.'),
    Document(content='Pipelines connect components via named ports.'),
]
store.write_documents(docs)
print(store.count_documents())

## 2. BM25 retrieval

In [ ]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
r = InMemoryBM25Retriever(document_store=store, top_k=2)
r.run(query='Haystack pipelines')

## 3. RAG pipeline

In [ ]:
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage

template = [ChatMessage.from_user('Context:\n{% for d in documents %}{{d.content}}\n{% endfor %}\nQ: {{query}}\nA:')]
pipe = Pipeline()
pipe.add_component('retriever', InMemoryBM25Retriever(document_store=store))
pipe.add_component('prompt', ChatPromptBuilder(template=template, required_variables=['query']))
pipe.add_component('llm', OpenAIChatGenerator(model='gpt-4o-mini'))
pipe.connect('retriever.documents', 'prompt.documents')
pipe.connect('prompt.prompt', 'llm.messages')

q = 'What is Haystack?'
pipe.run({'retriever': {'query': q}, 'prompt': {'query': q}})['llm']['replies'][0].text

## 4. Serialize/Deserialize

In [ ]:
yaml_str = pipe.dumps()
print(yaml_str[:500])
restored = Pipeline.loads(yaml_str)